<hr style="margin-bottom: 40px;">

<img src="https://user-images.githubusercontent.com/7065401/75165824-badf4680-5701-11ea-9c5b-5475b0a33abf.png"
    style="width:300px; float: right; margin: 0 40px 40px 40px;"></img>
    
# Unit 4. pandas Basics

pandas contains data structures and data manipulation tools designed to make data cleaning and analysis fast and convenient in Python. pandas is often used in tandem with numerical computing tools like NumPy and SciPy, analytical libraries like statsmodels and scikit-learn, and data visualization libraries like matplotlib.

While pandas adopts many coding idioms from NumPy, the biggest difference is that pandas is designed for working with **tabular or heterogeneous data**. NumPy, by contrast, is best suited for working with **homogeneously typed numerical array data**.

In [2]:
import numpy as np
import pandas as pd

## 4.1 pandas Series
pandas Series is similar to the 1-dimensional NumPy array, but associated with the special **index** and **array** attributes (the attribute index itself has an attribute **name**). Basic operations like creating Series, indexing and slicing, and array operations are almost identical.

Note that null values in pandas is **pd.NA** (compared to np.nan in NumPy), so functions like np.isnan() corresponds to pd.isna() in pandas.

In [3]:
# Method 1: from a python list
# series.index
series_1 = pd.Series([1, 2, 3, 4, 5])
series_1.index = list("abcde") # need to make sure index and values match in length
series_1.array
series_1.name = "Series 1" # name of this series
series_1

series_2 = pd.Series([1, 2, None, 4, 5], index=list("abcde"), name="Series 2")
series_2.index.name = "ID" # name of index
series_2

# Method 2: from a numpy list
np_1 = np.arange(5)
series_3 = pd.Series(np_1, index=list("abcde"), name="Series 3")
series_3

# Method 3: from a python dictionary
dict_1 = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5} # key as index, value as array
series_4 = pd.Series(dict_1, name="Series 4")
series_4

# Method 4: from another pandas series
series_5 = pd.Series(series_2, index=["c", "e", "a", "d", "f"]) # filtering
series_5

c    NaN
e    5.0
a    1.0
d    4.0
f    NaN
Name: Series 2, dtype: float64

In [4]:
# Indexing and slicing
# Think of pandas series as a mixture of python dictionary and list
print(series_5["a"]) # recommended
# print(series_5[2])
print(series_5[["a", "c"]])
print(series_5["e":"d"]) # the end is inclusive
print(series_5[series_5 > 2])

1.0
a    1.0
c    NaN
Name: Series 2, dtype: float64
e    5.0
a    1.0
d    4.0
Name: Series 2, dtype: float64
e    5.0
d    4.0
Name: Series 2, dtype: float64


In [5]:
# pd.NA is an extension data type
# NA actually stands for "Not Available"
# Three-value boolean operation: True, False, <NA> (unknown)
print(pd.NA == 1)
print(pd.NA == np.nan)
print(pd.NA is None)
# Just like np.isnan(), we have pd.isna() in pandas: treats None, np.nan, pd.NA all as null values
print(pd.isna(series_2))
print(pd.isna(pd.Series([1, 2, np.nan, 4, 5])))
print(pd.isna(pd.Series([1, 2, pd.NA, 4, 5])))
print(pd.notna(pd.Series([1, 2, pd.NA, 4, 5])))

<NA>
<NA>
False
ID
a    False
b    False
c     True
d    False
e    False
Name: Series 2, dtype: bool
0    False
1    False
2     True
3    False
4    False
dtype: bool
0    False
1    False
2     True
3    False
4    False
dtype: bool
0     True
1     True
2    False
3     True
4     True
dtype: bool


## 4.2 pandas DataFrame
pandas DataFrame can be thought of as a **dictionary of Series** all sharing the same index. It has both an **index** attribute (representing rows) and **columns** attribute (representing columns), both with **name**.

### 4.2.1 Creating pandas DataFrames
pandas DataFrames can be flexibly created from dictionaries, NumPy Arrays, or existing DataFrames.

In [6]:
# Method 1: from a python dictionary
# df.index, df.columns, df.name, df.index.name, df.columns.name
dict_2 = {"a":[1, "one", True],
         "b": [2, "two", False],
         "c": [3, "three", False],
         "d": [4, "four", True]}
df_1 = pd.DataFrame(dict_2)
df_1.index = ["row1", "row2", "row3"]
df_1.columns = ["col1", "col2", "col3", "col4"]
df_1.name = "Hello"
df_1.index.name = "row"
df_1.columns.name = "col"
df_1

# Method 2: from a python/numpy 2d array
np_2 = np.arange(9).reshape(3, 3)
df_2 = pd.DataFrame(np_2, index=["row1", "row2", "row3"], columns=["col1", "col2", "col3"])
df_2.index.name = "row"
df_2.columns.name = "col"
df_2

col,col1,col2,col3
row,,,
row1,0,1,2
row2,3,4,5
row3,6,7,8


### 4.2.2 Index objects
Index objects and multiindexing will be discussed in more detail in later chapters.

In [7]:
rows = pd.Index(["row1", "row2", "row3"], name="row")
cols = pd.Index(["col1", "col2", "col3"], name="col")
df_3 = pd.DataFrame(np_2, index=rows, columns=cols)
df_3

# Multi-indexing
multi_index = pd.MultiIndex.from_arrays([[1, 2, 3], [4, 5, 6]], names=["level1", "level2"])
df_4 = pd.DataFrame(np_2, index=multi_index, columns=cols)
df_4

,col,col1,col2,col3
level1,level2,,,
1,4,0,1,2
2,5,3,4,5
3,6,6,7,8


## 4.3 Essential pandas Functionality

### 4.3.1 Indexing and filtering
Accessing columns and rows need to be treated separately with: **data[column]**, **data.loc[row, column]**, **data.iloc[irow, icolumn]**.

In [8]:
gradebook = {"Sex": ["F", "F", "M", "F", "M", "M", "M"],
            "Class": ["Class3", "Class1", "Class5", "Class2", "Class1", "Class3", "Class3"],
            "Grade": [96, 78, 86, 95, 63, 52, 77]}
students = pd.Index(["Amy", "Alice", "Bob", "Cindy", "Derik", "Eric", "Eason"], name="Name")
data = pd.DataFrame(gradebook, index=students)
print(data)

# access columns: data[column name], data[[column names]]
data[["Class", "Grade"]]

# access rows: data.loc[row name], data.iloc[index]
data.loc[["Alice", "Eric"]]
data.iloc[[1, 5]]
data.loc["Alice":"Eric"] # inclusive
data.iloc[1:5] # exclusive
data.loc[data["Grade"] < 80] # select rows with condition

# access certain rows and columns combined: data.loc[row_name, col_name], data.iloc[row_index, col_index]
data.loc["Derik", "Grade"]
data.loc["Alice":"Eric", ["Class", "Grade"]]
data.iloc[:5, 1:]

      Sex   Class  Grade
Name                    
Amy     F  Class3     96
Alice   F  Class1     78
Bob     M  Class5     86
Cindy   F  Class2     95
Derik   M  Class1     63
Eric    M  Class3     52
Eason   M  Class3     77


,Class,Grade
Name,,
Amy,Class3,96
Alice,Class1,78
Bob,Class5,86
Cindy,Class2,95
Derik,Class1,63


### 4.3.2 Reindexing
Resetting index and columns with **data.reindex(index=, columns=)**. When indices don't exactly match, beware of pd.NA values.

In [9]:
# filtering & reordering
data2 = data.reindex(index=["Eric", "Amy", "Bob", "Derik", "Cindy"], columns=["Class", "Grade"])
data2
data3 = data.reindex(index=["Eric", "Amy", "Bob", "Derik", "Amanda"], columns=["Class", "Grade", "Contact"])
# data3.loc["Amanda"] = ["Class2", 90, "amanda@gmail.com"]
data3.loc["Amanda", "Grade"] = 90
data3

,Class,Grade,Contact
Name,,,
Eric,Class3,52.0,NaN
Amy,Class3,96.0,NaN
Bob,Class5,86.0,NaN
Derik,Class1,63.0,NaN
Amanda,NaN,90.0,NaN


### 4.3.3 Adding and dropping entries along axis
Adding rows can be a bit tricky (data.loc[len(data)] or pd.concat). Dropping can be done simply with data.drop.

In [13]:
# In general not recommended to alter the original data
# dropping entries from dataframes
data4 = data3.drop("Contact", axis="columns")
data4 = data4.drop("Amanda", axis="index")
# adding entries take element-wise operations
data4.loc["Cindy", "Class"] = "Class2"
data4.loc["Cindy", "Grade"] = 92
data4

,Class,Grade
Name,,
Eric,Class3,52.0
Amy,Class3,96.0
Bob,Class5,86.0
Derik,Class1,63.0
Cindy,Class2,92.0


### 4.3.4 Arithmetic and data alignment
Similar to NumPy with **broadcasting**--need to beware of alignment issues.

* Misalignments by default causes null values to be filled; can be changed through **fill_value**
* Operations between Series and DataFrame can be broadcasted through a given axis (by default through the **row**)

In [29]:
# Arithmetics: need to consider null values in general
# Series
ex_1 = pd.Series([1, 2, 4, np.nan, 8])
print(ex_1 * 2)
ex_2 = pd.Series([3, np.nan, 5, 9, 6])
print(ex_1 + ex_2)
ex_1.add(ex_2, fill_value=0)
ex_2.multiply(ex_1, fill_value=1)
ex_3 = pd.Series([1, 2, 4, 7, 8], index=list("abcde"))
ex_4 = pd.Series([3, 1, 5, 9, 6], index=list("bcdef"))
print(ex_3 + ex_4)
print(ex_3.add(ex_4, fill_value=0))
# DataFrames
df_1 = pd.DataFrame([[1, np.nan, 3],
                    [4, 5, 6],
                    [7, 8, 9]], index=list("abc"), columns=["one", "two", "three"])
df_2 = pd.DataFrame([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]], index=list("acd"), columns=["two", "one", "three"])
print(df_1 + df_2)
print(df_1.add(df_2, fill_value=0))

# Aggregate function (sum, mean): ignore the null values by default
print(ex_1.sum())
print(ex_2.sum(skipna=False))
print(df_1.mean()) # by default, aggregate along axis=0 (index)
print(df_1.mean(axis="columns", skipna=False)) # axis=1

0     2.0
1     4.0
2     8.0
3     NaN
4    16.0
dtype: float64
0     4.0
1     NaN
2     9.0
3     NaN
4    14.0
dtype: float64
a     NaN
b     5.0
c     5.0
d    12.0
e    17.0
f     NaN
dtype: float64
a     1.0
b     5.0
c     5.0
d    12.0
e    17.0
f     6.0
dtype: float64
    one  three   two
a   3.0    6.0   NaN
b   NaN    NaN   NaN
c  12.0   15.0  12.0
d   NaN    NaN   NaN
    one  three   two
a   3.0    6.0   1.0
b   4.0    6.0   5.0
c  12.0   15.0  12.0
d   8.0    9.0   7.0
15.0
nan
one      4.0
two      6.5
three    6.0
dtype: float64
a    NaN
b    5.0
c    8.0
dtype: float64


### 4.3.5 Function application and mapping
Functions (including self-defined and lambda functions) can be broadcasted element-wise along a given axis (by default **row**). Note the subtle difference between **data.apply(f)**, which performs axis-wise operations, and **data.map(f)**, which performs element-wise operations. A similar function **data.transform(f)** will be discussed along with data aggregation in later sections.

In [38]:
print(df_2.apply(lambda x: x.max() - x.min())) # x is an array/series
print(df_2.apply(lambda x: x.max() - x.min(), axis="columns"))
str_series = pd.Series(list("abaabba"))
print(str_series.map({"a": 1, "b": 2})) # dictionary for mapping
num_series = pd.Series(np.arange(5))
print(num_series.map(lambda x: x**2)) # x is an element
print(num_series.apply(lambda x: x**2))

two      6
one      6
three    6
dtype: int64
a    2
c    2
d    2
dtype: int64
0    1
1    2
2    1
3    1
4    2
5    2
6    1
dtype: int64
0     0
1     1
2     4
3     9
4    16
dtype: int64
0     0
1     1
2     4
3     9
4    16
dtype: int64


### 4.3.6 Sorting, ranking, and counting

* Sorting: unlike np.sort(), in pandas you can sort by both index (**data.sort_index**) and value along a given axis (**data.sort_values**)
* Ranking: **data.rank** gives the rank of a Series, including ties by default
* Counting: **data.value_counts** gives the count of each possible value or value combinations 

In [69]:
# Series
ex_5 = pd.Series([3, 4, 9, np.nan, 1, 6], index=list("bcaefd"))
print(ex_5.sort_index())
print(ex_5.sort_index(ascending=False))
print(ex_5.sort_values())
print(ex_5.sort_values(na_position="first"))
print(ex_5.sort_values(ascending=False, na_position="first"))

ex_6 = pd.Series([2, 4, 5, 2, 1, 2, 4, 8, 3])
print(ex_5.rank())
print(ex_5.rank(ascending=False))
print(ex_6.rank()) # by default: the average is taken among the ties
print(ex_6.rank(ascending=False, method="min"))
print(ex_6.rank(ascending=False, method="first"))

print(ex_6.value_counts())
print(ex_6.value_counts()[ex_6.value_counts() == 1]) # filtering the elements that appear only once

# DataFrames: indicate the row/col to sort by
ex_7 = pd.DataFrame([[6, 5, 4],
                    [1, 2, 3],
                    [9, 7, 8]], index=list("abc"), columns=["one", "two", "three"])
print(ex_7.sort_index(ascending=False))
print(ex_7.sort_values("a", axis="columns"))
print(ex_7.sort_values("three"))

a    9.0
b    3.0
c    4.0
d    6.0
e    NaN
f    1.0
dtype: float64
f    1.0
e    NaN
d    6.0
c    4.0
b    3.0
a    9.0
dtype: float64
f    1.0
b    3.0
c    4.0
d    6.0
a    9.0
e    NaN
dtype: float64
e    NaN
f    1.0
b    3.0
c    4.0
d    6.0
a    9.0
dtype: float64
e    NaN
a    9.0
d    6.0
c    4.0
b    3.0
f    1.0
dtype: float64
b    2.0
c    3.0
a    5.0
e    NaN
f    1.0
d    4.0
dtype: float64
b    4.0
c    3.0
a    1.0
e    NaN
f    5.0
d    2.0
dtype: float64
0    3.0
1    6.5
2    8.0
3    3.0
4    1.0
5    3.0
6    6.5
7    9.0
8    5.0
dtype: float64
0    6.0
1    3.0
2    2.0
3    6.0
4    9.0
5    6.0
6    3.0
7    1.0
8    5.0
dtype: float64
0    6.0
1    3.0
2    2.0
3    7.0
4    9.0
5    8.0
6    4.0
7    1.0
8    5.0
dtype: float64
2    3
4    2
5    1
1    1
8    1
3    1
Name: count, dtype: int64
5    1
1    1
8    1
3    1
Name: count, dtype: int64
   one  two  three
c    9    7      8
b    1    2      3
a    6    5      4
   three  two  one
a      4    

## 4.4 Descriptive Statistics with pandas
pandas objects are equipped with a set of common mathematical and statistical methods, most of which fall into the category of reductions or summary statistics. Unlike NumPy, pandas generally **handles (ignores) missing data** by default (need **skipna = false** to count null values).

### 4.4.1 General statistical methods
Similar to ndarray operations, with some cool methods like **data.describe()**. 

In [79]:
stockdata = pd.read_csv("stockdata.csv", parse_dates=["Date"])
stockdata.head() # by default first 5 rows
stockdata.head(10) # does the same thing data.iloc[:n]
stockdata.tail() # by default last 5 rows
stockdata.tail(10) # does the same thing data.iloc[-n:]
stockdata["MSFT"].isna().any()
stockdata.describe()
# stockdata["AAPL"].describe()

,MSFT,IBM,SBUX,AAPL,GSPC,Date
count,2306.000000,2306.000000,2306.000000,2306.000000,2306.000000,2306
mean,28.737785,136.239600,23.213089,54.719810,1459.999869,2011-07-31 21:25:45.533391104
min,12.630211,61.274273,3.284911,10.345740,676.530029,2007-01-03 00:00:00
25%,22.582537,102.407469,10.792787,22.929971,1198.417481,2009-04-17 18:00:00
50%,24.990737,143.833125,17.896952,49.327484,1402.370056,2011-07-30 12:00:00
75%,33.873379,172.566340,35.305682,78.102594,1770.579986,2013-11-12 18:00:00
max,56.146870,198.948796,63.098822,130.671320,2130.820068,2016-03-01 00:00:00
std,9.471737,37.376360,15.522634,34.645282,361.673516,NaN


### 4.4.2 Correlation and covariance
Compute correlation and covariance between two Series, where corr = cov/(std1 * std2). Will discuss later in modeling libraries in later sections.

In [82]:
stockdata["MSFT"].corr(stockdata["AAPL"])
stockdata["MSFT"].cov(stockdata["IBM"])
stockdata.corr()

,MSFT,IBM,SBUX,AAPL,GSPC,Date
MSFT,1.000000,0.482018,0.931701,0.868110,0.922886,0.812040
IBM,0.482018,1.000000,0.627626,0.742768,0.548516,0.795898
SBUX,0.931701,0.627626,1.000000,0.925332,0.894266,0.908655
AAPL,0.868110,0.742768,0.925332,1.000000,0.828452,0.941154
GSPC,0.922886,0.548516,0.894266,0.828452,1.000000,0.739568
Date,0.812040,0.795898,0.908655,0.941154,0.739568,1.000000


![purple-divider](https://user-images.githubusercontent.com/7065401/52071927-c1cd7100-2562-11e9-908a-dde91ba14e59.png)